# TeClass: Sarvam LLM with anti-bias + few-shot prompt

Combines two of the strongest prompt-engineering tricks observed so far:
- Anti-bias instruction (classes are balanced, don't default to HREL)
- Few-shot in-context examples (3 labelled pairs, one per class)

Model: sarvam-105b-conversations
Samples: 400 stratified test examples
Expected cost: ₹8
Expected runtime: 20 min

## 1. Setup

In [1]:
!pip install -q requests tqdm

In [2]:
import os
import re
import json
import time
import random
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, confusion_matrix

try:
    from google.colab import userdata
    SARVAM_KEY = userdata.get('SARVAM_API_KEY')
except Exception:
    SARVAM_KEY = os.environ.get('SARVAM_API_KEY', 'PASTE_SARVAM_KEY_HERE')

SARVAM_URL = "https://api.sarvam.ai/v1/chat/completions"
SARVAM_MODEL = "sarvam-105b-conversations"

r = requests.post(
    SARVAM_URL,
    headers={
        "api-subscription-key": SARVAM_KEY,
        "Content-Type": "application/json",
    },
    json={
        "model": SARVAM_MODEL,
        "messages": [{"role": "user", "content": "Say ok."}],
        "temperature": 0,
        "max_tokens": 50,
    },
    timeout=30,
)
print("status:", r.status_code)
if r.status_code == 200:
    print("sarvam:", r.json()["choices"][0]["message"]["content"].strip())
else:
    print("error:", r.text[:300])

status: 200
sarvam: Ok.


## 2. Load test + train data

In [3]:
def load(path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    art  = next(cols[c] for c in ["article", "text", "body", "news_article"] if c in cols)
    head = next(cols[c] for c in ["headline", "title", "news_headline"] if c in cols)
    lab  = next(cols[c] for c in ["label", "category", "class"] if c in cols)
    df = df[[art, head, lab]].rename(columns={art: "article", head: "headline", lab: "label"})
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    df = df[df["label"].isin(["HREL", "MREL", "LREL"])].reset_index(drop=True)
    return df

test_df = load("test.csv")
train_df = load("train.csv")

LABELS = ["HREL", "MREL", "LREL"]
lab2id = {"HREL": 0, "MREL": 1, "LREL": 2}

print("test:", test_df.shape, "| train:", train_df.shape)
print(test_df["label"].value_counts())

test: (3927, 3) | train: (18324, 3)
label
MREL    1523
HREL    1278
LREL    1126
Name: count, dtype: int64


## 3. Anti-bias + few-shot prompt

Combines the anti-bias instruction (that pushed zero-shot from 0.24 to 0.36) with 3 in-context examples.

In [4]:
DEFS = """IMPORTANT: Classes are approximately balanced in the test set
(roughly 1/3 HREL, 1/3 MREL, 1/3 LREL). Do NOT default to HREL.
Look carefully for LREL cues like sensational language or claims unsupported by the article.

Categories:
- HREL: headline is a faithful summary of the article's main event.
- MREL: headline is loosely related, covers a side event or weak inference.
- LREL: headline is misleading, sensational, clickbait, or draws unsupported conclusions."""

def cut(text, n=2200):
    text = str(text)
    return text if len(text) <= n else text[:n] + " [...]"

# Deterministic examples: first row of each class in train (same seed as other notebooks)
random.seed(42)
fewshot_ex = {lab: train_df[train_df["label"] == lab].iloc[0] for lab in LABELS}

def p_antibias_fewshot(a, h):
    ex = "\n\n".join(
        f"Article: {cut(row['article'], 500)}\nHeadline: {row['headline']}\nLabel: {lab}"
        for lab, row in fewshot_ex.items()
    )
    return (f"Classify the relevance between a Telugu news article and headline.\n\n"
            f"{DEFS}\n\n"
            f"Examples:\n\n{ex}\n\n"
            f"Now classify:\nArticle: {cut(a)}\n\nHeadline: {h}\n\n"
            f"Reply with ONE label only: HREL, MREL, or LREL.")

print(p_antibias_fewshot(test_df.iloc[0]["article"], test_df.iloc[0]["headline"])[:1200])

Classify the relevance between a Telugu news article and headline.

IMPORTANT: Classes are approximately balanced in the test set
(roughly 1/3 HREL, 1/3 MREL, 1/3 LREL). Do NOT default to HREL.
Look carefully for LREL cues like sensational language or claims unsupported by the article.

Categories:
- HREL: headline is a faithful summary of the article's main event.
- MREL: headline is loosely related, covers a side event or weak inference.
- LREL: headline is misleading, sensational, clickbait, or draws unsupported conclusions.

Examples:

Article: ప్రముఖ దర్శకులు శేఖర్ కమ్ముల ఇంట విషాదం నెలకొంది . ఆయన తండ్రి కమ్ముల శేషయ్య ( 89 ) కన్నుమూశారు . ఆయన గత కొద్దికాలంగా వృద్దాప్య సంబంధింత సమస్యలతో బాధపడుతున్నారు . ప్రస్తుత పరిస్థితుల్లో కొద్దికాలంగా ఇంటి వద్దే చికిత్స అందిస్తున్నారు . ఇటీవల ఆరోగ్యం క్షీణించడంతో స్థానిక ప్రైవేట్ హాస్పిటల్లో చేర్పించారు . అక్కడే చికిత్స పొందుతూ శనివారం ఉదయం 6 గంటలకు మరణించారు అని సన్నిహితులు తెలిపారు . శనివారం సాయంత్రం బన్సీలాల్ పేట స్మశాన వాటికలో అంత్యక్రియలు 

## 4. Sarvam wrapper + parser

In [5]:
LABEL_RE = re.compile(r"\b(HREL|MREL|LREL)\b", re.IGNORECASE)

def parse_label(text):
    if not text:
        return "MREL"
    try:
        obj = json.loads(text.strip())
        if isinstance(obj, dict) and "label" in obj:
            m = LABEL_RE.search(str(obj["label"]))
            if m:
                return m.group(1).upper()
    except Exception:
        pass
    m = LABEL_RE.search(text)
    if m:
        return m.group(1).upper()
    return "MREL"

def call_sarvam(prompt):
    resp = requests.post(
        SARVAM_URL,
        headers={
            "api-subscription-key": SARVAM_KEY,
            "Content-Type": "application/json",
        },
        json={
            "model": SARVAM_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
            "max_tokens": 200,
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"] or ""

def safe_call(fn, prompt, tries=5):
    for i in range(tries):
        try:
            return fn(prompt)
        except Exception as e:
            if i == tries - 1:
                return f"ERROR: {e}"
            wait = 4 * (2 ** i) + random.random()
            print(f"  retry {i+1}: {type(e).__name__}, waiting {wait:.1f}s")
            time.sleep(wait)

DELAY = 2.0

## 5. Runner

In [6]:
OUT = Path("llm_results_sarvam")
OUT.mkdir(exist_ok=True)

def run(df, n=None):
    n = n or len(df)
    path = OUT / f"{SARVAM_MODEL}__antibias_fewshot__n{n}.csv"
    if path.exists():
        cached = pd.read_csv(path)
        if len(cached) >= n:
            print(f"  using cache: {path.name}")
            return cached

    sub = df.head(n)
    rows = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc=f"{SARVAM_MODEL}/antibias_fewshot"):
        text = safe_call(call_sarvam, p_antibias_fewshot(r["article"], r["headline"]))
        rows.append({
            "true": r["label"],
            "pred": parse_label(text),
            "raw": (text or "")[:500],
        })
        if (i + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)
        time.sleep(DELAY)

    out = pd.DataFrame(rows)
    out.to_csv(path, index=False)
    return out

## 6. Smoke test (30 samples)

In [7]:
smoke = run(test_df, n=30)
print("accuracy:", (smoke["true"] == smoke["pred"]).mean())
print(smoke["pred"].value_counts())
print("\nfirst raw response:")
print(smoke["raw"].iloc[0])

sarvam-105b-conversations/antibias_fewshot:   0%|          | 0/30 [00:00<?, ?it/s]

accuracy: 0.3
pred
HREL    17
MREL    13
Name: count, dtype: int64

first raw response:
MREL

The headline frames the incident as a broader rhetorical question ("Is making noise a crime?") which sensationalizes the specific event of a headmistress tying up students. While the article describes the incident, the headline's phrasing is slightly misleading and clickbait-ish, drawing an unsupported broader conclusion about the criminality of making noise. It is not a direct summary (HREL) but leans toward being loosely related and sensational (MREL).


## 7. Full run (400 stratified samples)

In [8]:
N = 400
per_class = N // 3

eval_df = (test_df.groupby("label", group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), per_class), random_state=42))
                  .reset_index(drop=True))

print(f"running on {len(eval_df)} samples:")
print(eval_df["label"].value_counts())

results = run(eval_df, n=len(eval_df))
print("\ndone.")

running on 399 samples:
label
HREL    133
LREL    133
MREL    133
Name: count, dtype: int64


/tmp/ipykernel_4019/1599760429.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_class), random_state=42))


sarvam-105b-conversations/antibias_fewshot:   0%|          | 0/399 [00:00<?, ?it/s]


done.


## 8. Scores

In [9]:
yt = results["true"].map(lab2id).values
yp = results["pred"].map(lab2id).fillna(1).astype(int).values

f1_each = f1_score(yt, yp, labels=[0, 1, 2], average=None)
f1_w = f1_score(yt, yp, average="weighted")
f1_m = f1_score(yt, yp, average="macro")

print("Sarvam-105b / anti-bias fewshot results")
print("-" * 40)
print(f"HREL F1: {f1_each[0]:.3f}")
print(f"MREL F1: {f1_each[1]:.3f}")
print(f"LREL F1: {f1_each[2]:.3f}")
print(f"W F1:    {f1_w:.3f}")
print(f"M F1:    {f1_m:.3f}")

cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
print("\nconfusion matrix:")
print("           pred HREL  pred MREL  pred LREL")
for i, lab in enumerate(LABELS):
    print(f"true {lab}:  {cm[i,0]:>8}   {cm[i,1]:>8}   {cm[i,2]:>8}")

Sarvam-105b / anti-bias fewshot results
----------------------------------------
HREL F1: 0.618
MREL F1: 0.313
LREL F1: 0.163
W F1:    0.365
M F1:    0.365

confusion matrix:
           pred HREL  pred MREL  pred LREL
true HREL:       130          3          0
true MREL:        95         36          2
true LREL:        63         58         12


## 9. Comparison with baselines and other Sarvam prompts

In [10]:
ml_base   = {"HREL": 0.61, "MREL": 0.52, "LREL": 0.60, "W": 0.57, "M": 0.58}
bert_best = {"HREL": 0.68, "MREL": 0.57, "LREL": 0.64, "W": 0.62, "M": 0.63}
sarvam_zs    = {"HREL": 0.54, "MREL": 0.15, "LREL": 0.04, "W": 0.24, "M": 0.24}
sarvam_cot   = {"HREL": 0.54, "MREL": 0.12, "LREL": 0.11, "W": 0.26, "M": 0.26}
sarvam_fs    = {"HREL": 0.58, "MREL": 0.25, "LREL": 0.04, "W": 0.29, "M": 0.29}
sarvam_ab    = {"HREL": 0.59, "MREL": 0.27, "LREL": 0.22, "W": 0.36, "M": 0.36}

sarvam_abfs = {
    "HREL": round(f1_each[0], 2),
    "MREL": round(f1_each[1], 2),
    "LREL": round(f1_each[2], 2),
    "W":    round(f1_w, 2),
    "M":    round(f1_m, 2),
}

line = "=" * 96
print(line)
print("COMPARISON TABLE")
print(line)
print(f"{'method':<52} {'HREL':>6} {'MREL':>6} {'LREL':>6} {'W':>6} {'M':>6}")
print("-" * 96)

def pr(name, d):
    print(f"{name:<52} {d['HREL']:>6.2f} {d['MREL']:>6.2f} {d['LREL']:>6.2f} {d['W']:>6.2f} {d['M']:>6.2f}")

pr("ML baseline (LR, 5 features)", ml_base)
pr("Paper BERT (mDeBERTa)", bert_best)
print("-" * 96)
pr("Sarvam-105b / zeroshot", sarvam_zs)
pr("Sarvam-105b / cot", sarvam_cot)
pr("Sarvam-105b / fewshot", sarvam_fs)
pr("Sarvam-105b / anti-bias zeroshot", sarvam_ab)
pr("Sarvam-105b / anti-bias fewshot (new)", sarvam_abfs)
print(line)

COMPARISON TABLE
method                                                 HREL   MREL   LREL      W      M
------------------------------------------------------------------------------------------------
ML baseline (LR, 5 features)                           0.61   0.52   0.60   0.57   0.58
Paper BERT (mDeBERTa)                                  0.68   0.57   0.64   0.62   0.63
------------------------------------------------------------------------------------------------
Sarvam-105b / zeroshot                                 0.54   0.15   0.04   0.24   0.24
Sarvam-105b / cot                                      0.54   0.12   0.11   0.26   0.26
Sarvam-105b / fewshot                                  0.58   0.25   0.04   0.29   0.29
Sarvam-105b / anti-bias zeroshot                       0.59   0.27   0.22   0.36   0.36
Sarvam-105b / anti-bias fewshot (new)                  0.62   0.31   0.16   0.36   0.36
